In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [5]:
# Define paths
processed_clips_folder = "C:\\Users\\Fadilah Thasnim\\Desktop\\Academics\\7 - Semester\\8. TARP\\EmergencyCarParkModel\\ProcessedClips"
output_folder = "C:\\Users\\Fadilah Thasnim\\Desktop\\Academics\\7 - Semester\\8. TARP\\EmergencyCarParkModel\\OutputofTestingDataset"


In [6]:
# Build the 3D CNN Model
def build_3d_cnn(input_shape=(150, 64, 64, 3)):
    model = models.Sequential()
    model.add(layers.Conv3D(32, kernel_size=(3, 3, 5), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2)))
    
    model.add(layers.Conv3D(64, kernel_size=(3, 3, 3), activation='relu'))
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2)))
    
    model.add(layers.Conv3D(128, kernel_size=(3, 3, 3), activation='relu'))
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2)))
    
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))  # Binary classification

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [7]:
# Load Data
def load_data(folder):
    X, y = [], []
    for filename in os.listdir(folder):
        if filename.endswith(".npy"):
            clip_path = os.path.join(folder, filename)
            clip_data = np.load(clip_path)
            X.append(clip_data)
            
            # Assign labels based on filename convention (adjust based on actual labeling convention)
            label = 1 if "positive" in filename else 0
            y.append(label)
    
    X = np.array(X)
    y = np.array(y)
    return X, y

In [9]:
# Load data from the processed clips folder
X, y = load_data(processed_clips_folder)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Instantiate the model
model = build_3d_cnn()

In [12]:
# Train the model
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=4)

Epoch 1/5
82/82 [==============================] - 1702s 21s/step - loss: 0.8614 - accuracy: 0.6311 - val_loss: 0.6473 - val_accuracy: 0.6506
Epoch 2/5
82/82 [==============================] - 1323s 16s/step - loss: 0.6445 - accuracy: 0.6738 - val_loss: 0.6620 - val_accuracy: 0.6506
Epoch 3/5
82/82 [==============================] - 1200s 15s/step - loss: 0.6431 - accuracy: 0.6738 - val_loss: 0.6471 - val_accuracy: 0.6506
Epoch 4/5
82/82 [==============================] - 1196s 15s/step - loss: 0.6422 - accuracy: 0.6677 - val_loss: 0.6487 - val_accuracy: 0.6506
Epoch 5/5
82/82 [==============================] - 1193s 15s/step - loss: 0.6332 - accuracy: 0.6738 - val_loss: 0.6512 - val_accuracy: 0.6506


In [ ]:
# Save the model
model.save(os.path.join(output_folder, "3d_cnn_parking_model.h5"))

In [1]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=2)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

NameError: name 'model' is not defined

In [14]:
# Save predictions for the validation set
predictions = model.predict(X_val)
np.save(os.path.join(output_folder, "validation_predictions.npy"), predictions)

1/1 [==============================] - 1s 1s/step


In [2]:
model.save("C:\\Users\\Fadilah Thasnim\\Desktop\\Academics\\7 - Semester\\8. TARP\\EmergencyCarParkModel\\OutputofTestingDataset\\3d_cnn_parking_model")


NameError: name 'model' is not defined

In [81]:
import cv2
import numpy as np
import tensorflow as tf

# Preprocess a single video clip and save it as an .npy file
def preprocess_and_save_video_clip(video_path, output_path, frame_count=150, frame_size=(64, 64)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    for _ in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, frame_size)
        frames.append(frame)
    
    cap.release()
    clip_data = np.array(frames) / 255.0  # Normalize frames

    # Save as .npy file
    np.save(output_path, clip_data)
    print(f"Video clip saved as {output_path}")

# Load and predict on an .npy file
def load_and_predict_from_npy(npy_path, model):
    new_clip = np.load(npy_path)  # Load the .npy file
    new_clip = np.expand_dims(new_clip, axis=0)  # Add batch dimension

    # Predict the class of the new video
    prediction = model.predict(new_clip)

    # Convert prediction to binary label
    binary_label = 1 if prediction > 0.3 else 0

    # Display results
    print(f"Prediction Probability: {prediction[0][0]:.4f}")
    print(f"Class Label (0: Negative, 1: Positive): {binary_label}")

# Paths to the video and output npy file
new_video_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video24_positive.mp4"  # Replace with actual path
output_npy_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video24_positive.npy"

# Preprocess and save the video as an .npy file
preprocess_and_save_video_clip(new_video_path, output_npy_path)

# Load the TensorFlow SavedModel
model_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\OutputofTestingDataset\3d_cnn_parking_model.h5"
model = tf.keras.models.load_model(model_path)

# Load the .npy file and make predictions
load_and_predict_from_npy(output_npy_path, model)


Video clip saved as C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video24_positive.npy
1/1 [==============================] - 0s 379ms/step
Prediction Probability: 0.3938
Class Label (0: Negative, 1: Positive): 1


In [82]:
# Load the trained model
model = tf.keras.models.load_model(model_path)

In [83]:
def load_video_for_display(video_path, frame_count=150, frame_size=(64, 64)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    original_frames = []
    
    for _ in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            break
        resized_frame = cv2.resize(frame, frame_size)
        frames.append(resized_frame / 255.0)  # Normalize for model input
        original_frames.append(frame)  # Keep original for display

    cap.release()
    return np.array(frames), original_frames

In [84]:
def predict_and_display_spot(video_path):
    frames, original_frames = load_video_for_display(video_path)
    frames = frames.reshape(1, *frames.shape)  # Model expects batch dimension
    prediction = model.predict(frames)[0][0]  # Assuming binary output
    
    if prediction > 0.2:  # Model predicts positive
        print("Vacant parking spot detected!")
        # Draw a circle on each original frame as an approximation
        for frame in original_frames:
            height, width, _ = frame.shape
            # Mark an approximate parking spot position
            cv2.circle(frame, (int(width * 0.8), int(height * 0.5)), 20, (0, 255, 0), 3)
            cv2.imshow("Vacant Parking Spot Detected", frame)
            cv2.waitKey(100)  # Display each frame briefly

    else:
        print("No vacant parking spot detected in the clip.")
    
    cv2.destroyAllWindows()


In [85]:
# Run the function on a test video
predict_and_display_spot(r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video24_positive.mp4")

1/1 [==============================] - 0s 445ms/step
Vacant parking spot detected!


In [87]:
import cv2
import numpy as np
import tensorflow as tf

# Preprocess a single video clip and get frames for marking
def preprocess_video_clip_for_marking(video_path, frame_count=150, frame_size=(64, 64)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    original_frames = []
    
    for _ in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            break
        resized_frame = cv2.resize(frame, frame_size)
        frames.append(resized_frame)
        original_frames.append(frame)  # Store original frames for marking
    
    cap.release()
    clip_data = np.array(frames) / 255.0  # Normalize frames
    return clip_data, original_frames

# Mark parking spots on each frame if video is classified as positive
def mark_and_save_video(original_frames, output_video_path, is_positive, mark_color=(0, 255, 0), mark_thickness=2):
    height, width, _ = original_frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, 20.0, (width, height))

    for frame in original_frames:
        if is_positive:
            # Example: Draw a circle on each frame to indicate parking space detection
            cv2.circle(frame, (width // 2, height // 2), 50, mark_color, mark_thickness)
        out.write(frame)

    out.release()
    print(f"Marked video saved at: {output_video_path}")

# Paths to the video and output mp4 file
new_video_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video24_positive.mp4"
output_video_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video25_marked.mp4"

# Load the TensorFlow SavedModel
model_path = r"C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\OutputofTestingDataset\3d_cnn_parking_model.h5"
model = tf.keras.models.load_model(model_path)

# Preprocess the video, predict, and save marked video
clip_data, original_frames = preprocess_video_clip_for_marking(new_video_path)
clip_data = np.expand_dims(clip_data, axis=0)  # Add batch dimension

# Predict the class of the new video
prediction = model.predict(clip_data)
binary_label = 1 if prediction > 0.3 else 0  # Class label (1: Positive, 0: Negative)

# Display results
print(f"Prediction Probability: {prediction[0][0]:.4f}")
print(f"Class Label (0: Negative, 1: Positive): {binary_label}")

# Mark frames and save as a new video
mark_and_save_video(original_frames, output_video_path, is_positive=(binary_label == 1))


1/1 [==============================] - 0s 366ms/step
Prediction Probability: 0.3938
Class Label (0: Negative, 1: Positive): 1
Marked video saved at: C:\Users\Fadilah Thasnim\Desktop\Academics\7 - Semester\8. TARP\EmergencyCarParkModel\Validation\video25_marked.mp4
